# Feature Engineering

Feature engineering is an important step because the original variables can be transformed into more informative features that help improve prediction accuracy.


## Import Libraries

In [28]:
import pandas as pd
import numpy as np

## Load the Dataset

In [29]:
df = pd.read_csv("../data/clean_data.csv")
original_len = len(df)
df.head()

,record_key,carrier_code,start_time,end_time,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,save_forward_marker,elapsed_seconds
0,rid2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,rid2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,rid3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
3,rid2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435
4,rid0801584,2,2016-01-30 22:01:40,2016-01-30 22:09:03,6,-73.982857,40.742195,-73.992081,40.749184,N,443


## Convert Time Features

In [30]:
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

In [31]:
# Start Hour
df['start_hour'] = df['start_time'].dt.hour

# Day of Week: values range from 0 (Monday) to 6 (Sunday)
df['day_of_week'] = df['start_time'].dt.dayofweek

# Month
df['month'] = df['start_time'].dt.month

# Weekend Indicator: if 1 = Weekend and if 0 = Weekday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df.head()

,record_key,carrier_code,start_time,end_time,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,save_forward_marker,elapsed_seconds,start_hour,day_of_week,month,is_weekend
0,rid2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,0,3,0
1,rid2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,6,6,1
2,rid3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429,19,2,4,0
3,rid2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435,13,5,3,1
4,rid0801584,2,2016-01-30 22:01:40,2016-01-30 22:09:03,6,-73.982857,40.742195,-73.992081,40.749184,N,443,22,5,1,1


## Create Rush Hour Indicator

Rush-hour periods are associated with increased traffic congestion and often lead to longer trip durations. Two approaches can be used to create this feature.

### Approach 1: Fixed Rush Hours

This approach uses commonly accepted urban traffic patterns.

- Morning rush hour: 7 AM – 9 AM
- Evening rush hour: 4 PM – 6 PM

- 1 = Rush Hour
- 0 = Non-Rush Hour

This method is simple and easy to interpret, but it relies on predefined assumptions about traffic patterns.


In [32]:
df['is_rush_hour'] = (
    ((df['start_hour'] >= 7) & (df['start_hour'] <= 9)) |
    ((df['start_hour'] >= 16) & (df['start_hour'] <= 18))
).astype(int)

df.head()

,record_key,carrier_code,start_time,end_time,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,save_forward_marker,elapsed_seconds,start_hour,day_of_week,month,is_weekend,is_rush_hour
0,rid2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,0,3,0,1
1,rid2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,6,6,1,0
2,rid3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429,19,2,4,0,0
3,rid2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435,13,5,3,1,0
4,rid0801584,2,2016-01-30 22:01:40,2016-01-30 22:09:03,6,-73.982857,40.742195,-73.992081,40.749184,N,443,22,5,1,1,0


## Calculate Geographical Distances

Calculate both Haversine and Manhattan distances in kilometers.


In [33]:
# Convert coordinates from degrees to radians
lon1 = np.radians(df['start_xcoord'])
lat1 = np.radians(df['start_ycoord'])

lon2 = np.radians(df['end_xcoord'])
lat2 = np.radians(df['end_ycoord'])

# Differences
dlon = lon2 - lon1
dlat = lat2 - lat1

# Haversine formula
a = (
    np.sin(dlat / 2) ** 2 +
    np.cos(lat1) * np.cos(lat2) *
    np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Earth's radius (km)
df['trip_distance_km'] = 6371 * c

# Approximate Manhattan distance
lat_distance = np.abs(df['start_ycoord'] - df['end_ycoord']) * 111.0
lon_distance = np.abs(df['start_xcoord'] - df['end_xcoord']) * 85.0

df['manhattan_distance_km'] = lat_distance + lon_distance


df[['trip_distance_km', 'manhattan_distance_km']].head()

,trip_distance_km,manhattan_distance_km
0,1.498521,1.748741
1,1.805507,2.443325
2,1.485498,1.660362
3,1.188588,1.197479
4,1.098942,1.559761


## Encode Categorical Features
Encode categorical variables into binary numerical values.

Since carrier_code contains only two unique categories (1 and 2), Binary Encoding was applied by mapping 1 → 0 and 2 → 1. This converts the categorical feature into a binary numerical feature suitable for machine learning algorithms while preserving all available information


In [34]:
# Display original values
print("\nOriginal values:")
print(df['carrier_code'].value_counts().sort_index())
print(df['save_forward_marker'].value_counts().sort_index())

# Encode save_forward_marker
if 'save_forward_marker' in df.columns:
    df['save_forward_marker'] = (
        df['save_forward_marker']
        .map({'N': 0, 'Y': 1})
        .astype(int)
    )

# Encode carrier_code
if 'carrier_code' in df.columns:
    df['carrier_code'] = (
        df['carrier_code']
        .map({1: 0, 2: 1})
        .astype(int)
    )


# Display encoded values
print("\nEncoded values:")
print(df['carrier_code'].value_counts().sort_index())
print(df['save_forward_marker'].value_counts().sort_index())

df[['carrier_code', 'save_forward_marker']].head()


Original values:
carrier_code
1    645809
2    738615
Name: count, dtype: int64
save_forward_marker
N    1377360
Y       7064
Name: count, dtype: int64

Encoded values:
carrier_code
0    645809
1    738615
Name: count, dtype: int64
save_forward_marker
0    1377360
1       7064
Name: count, dtype: int64


,carrier_code,save_forward_marker
0,1,0
1,0,0
2,1,0
3,1,0
4,1,0


In [ ]:
df.to_csv("../data/feature_engineered_data.csv", index=False)